In [2]:
import pyspark
from utils import load_config
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType
from pyspark.sql.functions import col, from_json, when, expr, round

class SparkStructuredStreaming:
    def __init__(self, config_file='config.json'):
        self.config = load_config(config_file)
        self.spark = self.setup_spark_session()
        self.define_schemas()

    def setup_spark_session(self):
        spark_version = pyspark.__version__
        session = (
            SparkSession.builder
            .appName("CentralKitchenPipeline")
            .config("spark.jars.packages", f"org.apache.spark:spark-sql-kafka-0-10_2.13:{spark_version}")
            .getOrCreate()
        )
        session.sparkContext.setLogLevel("OFF")
        return session

    def define_schemas(self):
        ingredient_schema = StructType([
            StructField("item", StringType(), True),
            StructField("amount_kg", DoubleType(), True)
        ])

        metadata_schema = StructType([
            StructField("event_id", StringType(), True),
            StructField("event_type", StringType(), True),
            StructField("source", StringType(), True),
            StructField("generated_at", StringType(), True)
        ])

        kitchen_payload_schema = StructType([
            StructField("batch_id", StringType(), True),
            StructField("station_id", StringType(), True),
            StructField("recipe_id", StringType(), True),
            StructField("action", StringType(), True),
            StructField("weight_kg", DoubleType(), True),
            StructField("ingredients", ArrayType(ingredient_schema), True),
            StructField("temperature_celsius", DoubleType(), True),
            StructField("event_timestamp", StringType(), True)
        ])

        dispatch_payload_schema = StructType([
            StructField("batch_id", StringType(), True),
            StructField("canteen_id", StringType(), True),
            StructField("driver_id", StringType(), True),
            StructField("action", StringType(), True),
            StructField("truck_temp_celsius", DoubleType(), True),
            StructField("event_timestamp", StringType(), True)
        ])

        self.kitchen_envelope = StructType([
            StructField("metadata", metadata_schema, True),
            StructField("payload", kitchen_payload_schema, True)
        ])

        self.dispatch_envelope = StructType([
            StructField("metadata", metadata_schema, True),
            StructField("payload", dispatch_payload_schema, True)
        ])

    def read_kafka_topic(self, servers, topic):
        return (
            self.spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", servers)
            .option("subscribe", topic)
            .load()
        )

    def parse_envelope(self, df, schema):
        return (
            df.selectExpr("CAST(value AS STRING)")
            .select(from_json(col("value"), schema).alias("data"))
            .select("data.metadata.*", "data.payload.*")
        )

    def process_dlq_stream(self, df):
        return (df.selectExpr(
            "CAST(key AS STRING) as message_key",
            "CAST(value AS STRING) as raw_message",
            "timestamp as event_timestamp"
        ))

    def apply_kitchen_business_rules(self, df):
        min_temp = self.config.get("min_cook_temp", 75.0)
        min_drop = self.config.get("min_weight_drop", 2.0)

        df_calc = df.withColumn(
            "expected_start_weight",
            round(expr("aggregate(ingredients, 0D, (acc, item) -> acc + item.amount_kg)"), 2)
        )
        
        return df_calc.withColumn(
            "sensor_status",
            when(col("weight_kg").isNull() | col("temperature_celsius").isNull(), "SENSOR_FAILURE").otherwise("OK")
        ).withColumn(
            "temperature_status",
            when((col("action") == "COOKING") & (col("temperature_celsius") < min_temp), "LOW_HEAT_WARNING").otherwise("OK")
        ).withColumn(
            "weight_status",
            when(col("weight_kg").isNull(), "UNKNOWN")
            .when((col("action").isin("PREPARING", "COOKING", "PACKING")) & 
                  ((col("expected_start_weight") - col("weight_kg")) >= min_drop), "SUSPICIOUS_DROP")
            .otherwise("OK")
        )

    def apply_dispatch_business_rules(self, df):
        max_temp = self.config.get("max_truck_temp", 5.0)
        return df.withColumn(
            "sensor_status",
            when(col("truck_temp_celsius").isNull(), "SENSOR_FAILURE").otherwise("OK")
        ).withColumn(
            "temperature_status",
            when(col("truck_temp_celsius") > max_temp, "WARM_TRUCK_WARNING").otherwise("OK")
        )

    def cleanup_resources(self, kitchen_q=None, dispatch_q=None, dlq_q=None):
        print("Starting resource cleanup...")
        
        for query in [kitchen_q, dispatch_q, dlq_q]:
            if query and query.isActive:
                print(f"Stopping stream query: {query.id}")
                query.stop()
        
        if self.spark:
            print("Closing Spark Session...")
            self.spark.stop()
        
        print("Cleanup complete.")

    def start_streaming(self):
        servers = self.config.get("kafka_bootstrap_servers", "localhost:9092")
        
        kitchen_topic = self.config.get("clean_kitchen_events", "clean_kitchen_events")
        kitchen_raw = self.read_kafka_topic(servers, kitchen_topic)
        kitchen_clean = self.apply_kitchen_business_rules(self.parse_envelope(kitchen_raw, self.kitchen_envelope))
        kitchen_query = (
            kitchen_clean.writeStream.format("parquet") 
            .option("path", self.config.get("kitchen_data_path", "./output/kitchen_data")) 
            .option("checkpointLocation", self.config.get("kitchen_checkpoint_path", "kitchen_checkpoint_path")).start()
        )

        dispatch_topic = self.config.get("clean_dispatch_events", "clean_dispatch_events")
        dispatch_raw = self.read_kafka_topic(servers, dispatch_topic)
        dispatch_clean = self.apply_dispatch_business_rules(self.parse_envelope(dispatch_raw, self.dispatch_envelope))
        dispatch_query = (
            dispatch_clean.writeStream.format("parquet")
            .option("path", self.config.get("dispatch_data_path", "./output/dispatch_data"))
            .option("checkpointLocation", self.config.get("dispatch_checkpoint_path", "dispatch_checkpoint_path")).start()
        )

        dlq_topic = self.config.get("dlq_topic", "dead_letter_queue")
        dlq_raw = self.read_kafka_topic(servers, dlq_topic)
        dlq_clean = self.process_dlq_stream(dlq_raw)
        dlq_query = (
            dlq_clean.writeStream.format("parquet")
            .option("path", self.config.get("dlq_data_path", "./output/dlq_data"))
            .option("checkpointLocation", self.config.get("dlq_checkpoint_path", "./output/checkpoints/dlq"))
            .start())

        print(f"Streaming is active. Monitoring for data in {kitchen_topic}, {dispatch_topic}, and {dlq_topic}.")

        try:
            self.spark.streams.awaitAnyTermination()
        except Exception as error:
            print(f"A stream error occurred: {error}")
        finally:
            self.cleanup_resources(kitchen_query, dispatch_query, dlq_query)

if __name__ == "__main__":
    stream = SparkStructuredStreaming()
    stream.start_streaming()

Streaming is active. Monitoring for data in clean_kitchen_events, clean_dispatch_events, and dead_letter_queue.


ERROR:root:Exception while sending command.                                     
Traceback (most recent call last):
  File "/home/student/de-venv/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
RuntimeError: reentrant call inside <_io.BufferedReader name=68>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/student/de-venv/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/student/de-venv/lib/python3.10/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/student/de-venv/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    an

A stream error occurred: An error occurred while calling o286.awaitAnyTermination
Starting resource cleanup...
Stopping stream query: 96185aa1-ca21-4598-b6f4-82b1f6cfd6be
Stopping stream query: 9292cfc0-634d-44a6-b36d-23729b53ced1
Stopping stream query: f67b2a35-c64f-4e34-826d-89e2b0767b0b
Closing Spark Session...
Cleanup complete.


In [10]:
import pyspark
from utils import load_config
from pyspark.sql import SparkSession

class DeadLetterMonitor:
    def __init__(self, config_file='config.json'):
        self.config = load_config(config_file)
        self.spark = self.setup_spark_session()

    def setup_spark_session(self):
        spark_version = pyspark.__version__
        session = (
            SparkSession.builder
            .appName("DLQViewer")
            .config("spark.jars.packages", f"org.apache.spark:spark-sql-kafka-0-10_2.13:{spark_version}")
            .getOrCreate()
        )
        return session

    def display_failures(self):
        servers = self.config.get("kafka_bootstrap_servers", "localhost:9092")
        dlq_topic = self.config.get("dlq_topic", "dead_letter_queue")

        print(f"Fetching messages from: {dlq_topic}")

        df = (
            self.spark.read
            .format("kafka")
            .option("kafka.bootstrap.servers", servers)
            .option("subscribe", dlq_topic)
            .option("startingOffsets", "earliest")
            .load()
        )

        # Convert binary data to text
        logs = df.selectExpr(
            "CAST(key AS STRING) as message_key", 
            "CAST(value AS STRING) as error_payload", 
            "timestamp"
        )
        
        logs.show(truncate=False)

if __name__ == "__main__":
    monitor = DeadLetterMonitor()
    monitor.display_failures()

Fetching messages from: dead_letter_queue
+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------+
|message_key |error_payload                                                                                                                                                                                                                                                                                                                                                                                                       

In [11]:
from utils import load_config
from pyspark.sql import SparkSession

def preview_parquet_data(spark_session, path, dataset_name):
    """Safely reads and prints the latest data from a Parquet directory."""
    print(f"\n=== Reading {dataset_name} Data ===")
    try:
        df = spark_session.read.parquet(path)
        # Sort by time so you can see the batch progress
        df.orderBy("event_timestamp").show(200, truncate=False)
    except Exception as e:
        print(f"No {dataset_name.lower()} data found at: {path}")

if __name__ == "__main__":
    # 1. Load Configuration
    config = load_config()
    kitchen_path = config.get("kitchen_data_path", "./output/kitchen_data")
    dispatch_path = config.get("dispatch_data_path", "./output/dispatch_data")

    # 2. Start Spark Session
    spark = SparkSession.builder \
        .appName("ReadSavedData") \
        .getOrCreate()
        
    spark.sparkContext.setLogLevel("WARN")

    # 3. Preview the Data
    preview_parquet_data(spark, kitchen_path, "Kitchen")
    preview_parquet_data(spark, dispatch_path, "Dispatch")
    
    # Optional: Stop the Spark session cleanly when done
    spark.stop()


=== Reading Kitchen Data ===
+------------+--------------+--------------+--------------------------+----------+----------+------------+---------+---------+-------------------------------------------------------------------+-------------------+----------------+---------------------+--------------+------------------+---------------+
|event_id    |event_type    |source        |generated_at              |batch_id  |station_id|recipe_id   |action   |weight_kg|ingredients                                                        |temperature_celsius|event_timestamp |expected_start_weight|sensor_status |temperature_status|weight_status  |
+------------+--------------+--------------+--------------------------+----------+----------+------------+---------+---------+-------------------------------------------------------------------+-------------------+----------------+---------------------+--------------+------------------+---------------+
|KIT-EVT-1001|kitchen_action|data_generator|2026-03-28T13: